# Atta Mills Phase 2 GPU MLP Walk-forward

Kaggle-ready notebook for the clean pre-match Premier League experiment.

- Target: full-time H/D/A result only.
- Data: football-data.co.uk Premier League CSVs.
- Features: 44 Atta Mills-style features + 16 rolling shot-efficiency features + 5 H2H features.
- Excludes from training: odds, half-time variables, O/U 2.5, FootyStats.
- Benchmark: Bet365 closing odds, de-vigged to implied probabilities.

On Kaggle, enable GPU in Notebook settings. Then add this project folder or a dataset containing `PL1920.csv` ... `PL2526.csv`.

In [ ]:
from pathlib import Path
import json, math, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, log_loss
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', DEVICE)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
PL_FILES = ['PL1920.csv','PL2021.csv','PL2122.csv','PL2223.csv','PL2324.csv','PL2425.csv','PL2526.csv']
CLASS_LABELS = [0, 1, 2]  # A, D, H
WINDOWS = [5, 10]
OUTPUT_DIR = Path('/kaggle/working/atta_mills_phase2_gpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMN_MAP = {
    'Div':'division','Date':'date','Time':'time','HomeTeam':'home_team','AwayTeam':'away_team',
    'FTHG':'home_goals_full','FTAG':'away_goals_full','FTR':'result_full',
    'HTHG':'home_goals_half','HTAG':'away_goals_half','HTR':'result_half','Referee':'referee',
    'HS':'home_shots','AS':'away_shots','HST':'home_shots_target','AST':'away_shots_target',
    'B365H':'odds_home','B365D':'odds_draw','B365A':'odds_away',
    'B365CH':'odds_home_close','B365CD':'odds_draw_close','B365CA':'odds_away_close',
}

def season_from_date(date):
    y = int(date.year)
    return f'{y}-{str(y + 1)[-2:]}' if int(date.month) >= 8 else f'{y - 1}-{str(y)[-2:]}'

def find_pl_files():
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]
    found = {}
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob('PL*.csv'):
            if path.name in PL_FILES and path.name not in found:
                found[path.name] = path
    missing = [f for f in PL_FILES if f not in found]
    if missing:
        raise FileNotFoundError(f'Missing PL CSVs: {missing}. Add them as a Kaggle dataset/input.')
    return [found[f] for f in PL_FILES]

def load_one(path):
    df = pd.read_csv(path)
    df = df.rename(columns=COLUMN_MAP)
    keep = [c for c in COLUMN_MAP.values() if c in df.columns]
    df = df[keep].copy()
    df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    for col in ['home_goals_full','away_goals_full','home_shots','away_shots','home_shots_target','away_shots_target',
                'odds_home_close','odds_draw_close','odds_away_close']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['target_result'] = df['result_full'].map({'A':0,'D':1,'H':2})
    df['source_file'] = path.name
    return df

paths = find_pl_files()
df = pd.concat([load_one(p) for p in paths], ignore_index=True).sort_values('date').reset_index(drop=True)
df['season_id'] = df['date'].apply(season_from_date)
print(paths)
print(df.shape)
print(df['season_id'].value_counts().sort_index())

In [ ]:
def empty_stats():
    return {'gf': [], 'ga': [], 'points': [], 'result': [], 'goal_diff': [], 'win_margin': [], 'loss_margin': [],
            'shots': [], 'sot': []}

def rolling_mean(values, window):
    return float(np.mean(values[-window:])) if values else np.nan

def rate(values, window, target):
    return float(np.mean([v == target for v in values[-window:]])) if values else np.nan

def team_features(stats, prefix, window, league_avg_goals):
    avg_gf = rolling_mean(stats['gf'], window)
    avg_ga = rolling_mean(stats['ga'], window)
    avg_shots = rolling_mean(stats['shots'], window)
    avg_sot = rolling_mean(stats['sot'], window)
    shot_accuracy = avg_sot / avg_shots if avg_shots and not math.isnan(avg_shots) else np.nan
    goals_sum = np.sum(stats['gf'][-window:]) if stats['gf'] else np.nan
    shots_sum = np.sum(stats['shots'][-window:]) if stats['shots'] else np.nan
    conversion = goals_sum / shots_sum if shots_sum and not math.isnan(shots_sum) else np.nan
    return {
        f'{prefix}_team_state_points_{window}': rolling_mean(stats['points'], window),
        f'{prefix}_attack_strength_{window}': avg_gf / league_avg_goals if league_avg_goals and not math.isnan(avg_gf) else np.nan,
        f'{prefix}_defense_strength_{window}': avg_ga / league_avg_goals if league_avg_goals and not math.isnan(avg_ga) else np.nan,
        f'{prefix}_goals_for_{window}': avg_gf,
        f'{prefix}_goals_against_{window}': avg_ga,
        f'{prefix}_goal_differential_{window}': rolling_mean(stats['goal_diff'], window),
        f'{prefix}_win_rate_{window}': rate(stats['result'], window, 1.0),
        f'{prefix}_draw_rate_{window}': rate(stats['result'], window, 0.0),
        f'{prefix}_loss_rate_{window}': rate(stats['result'], window, -1.0),
        f'{prefix}_win_margin_goals_{window}': rolling_mean(stats['win_margin'], window),
        f'{prefix}_loss_margin_goals_{window}': rolling_mean(stats['loss_margin'], window),
        f'{prefix}_avg_shots_{window}': avg_shots,
        f'{prefix}_avg_sot_{window}': avg_sot,
        f'{prefix}_shot_accuracy_{window}': shot_accuracy,
        f'{prefix}_conversion_rate_{window}': conversion,
    }

def build_features(data):
    histories = defaultdict(empty_stats)
    pair_histories = defaultdict(list)
    league_goals = []
    rows = []
    for _, m in data.sort_values('date').iterrows():
        h, a = m['home_team'], m['away_team']
        league_avg = float(np.mean(league_goals)) if league_goals else np.nan
        feats = {}
        for w in WINDOWS:
            feats.update(team_features(histories[h], 'home', w, league_avg))
            feats.update(team_features(histories[a], 'away', w, league_avg))
        pair = tuple(sorted([h, a]))
        prev = pair_histories[pair][-5:]
        feats['h2h_home_wins'] = sum(1 for x in prev if x['winner'] == h)
        feats['h2h_away_wins'] = sum(1 for x in prev if x['winner'] == a)
        feats['h2h_draws'] = sum(1 for x in prev if x['winner'] == 'D')
        feats['h2h_home_avg_goals'] = float(np.mean([x['home_goals'] if x['home'] == h else x['away_goals'] for x in prev])) if prev else 0.0
        feats['h2h_away_avg_goals'] = float(np.mean([x['away_goals'] if x['away'] == a else x['home_goals'] for x in prev])) if prev else 0.0
        rows.append(feats)

        hg, ag = int(m['home_goals_full']), int(m['away_goals_full'])
        hs, ast = m.get('home_shots', np.nan), m.get('away_shots', np.nan)
        hst, asot = m.get('home_shots_target', np.nan), m.get('away_shots_target', np.nan)
        for team, gf, ga, pts, res, shots, sot in [
            (h, hg, ag, 3 if hg > ag else 1 if hg == ag else 0, 1.0 if hg > ag else 0.0 if hg == ag else -1.0, hs, hst),
            (a, ag, hg, 3 if ag > hg else 1 if hg == ag else 0, 1.0 if ag > hg else 0.0 if hg == ag else -1.0, ast, asot),
        ]:
            diff = gf - ga
            histories[team]['gf'].append(float(gf)); histories[team]['ga'].append(float(ga))
            histories[team]['points'].append(float(pts)); histories[team]['result'].append(float(res))
            histories[team]['goal_diff'].append(float(diff))
            histories[team]['win_margin'].append(float(max(diff, 0))); histories[team]['loss_margin'].append(float(max(-diff, 0)))
            histories[team]['shots'].append(float(shots) if pd.notna(shots) else np.nan)
            histories[team]['sot'].append(float(sot) if pd.notna(sot) else np.nan)
        pair_histories[pair].append({'home': h, 'away': a, 'home_goals': hg, 'away_goals': ag, 'winner': h if hg > ag else a if ag > hg else 'D'})
        league_goals.extend([float(hg), float(ag)])
    feat = pd.DataFrame(rows)
    return pd.concat([data.reset_index(drop=True), feat], axis=1), list(feat.columns)

df_feat, feature_cols = build_features(df)
print('features:', len(feature_cols))
pd.Series(feature_cols).to_csv(OUTPUT_DIR / 'feature_columns.csv', index=False)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim); self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim); self.bn2 = nn.BatchNorm1d(dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.drop(F.relu(self.bn1(self.fc1(x))))
        h = self.drop(self.bn2(self.fc2(h)))
        return F.relu(x + h)

class MLPDeep(nn.Module):
    def __init__(self, in_dim, hidden=128, dropout=0.30):
        super().__init__()
        self.input = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU())
        self.blocks = nn.Sequential(*[ResidualBlock(hidden, dropout) for _ in range(4)])
        self.head = nn.Linear(hidden, 3)
    def forward(self, x):
        return self.head(self.blocks(self.input(x)))

class MLPWide(nn.Module):
    def __init__(self, in_dim, dropout=0.30):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 3))
    def forward(self, x): return self.net(x)

MODELS = {'mlp_deep': MLPDeep, 'mlp_wide': MLPWide}

def market_probs(part):
    odds = part[['odds_home_close','odds_draw_close','odds_away_close']].to_numpy(float)
    probs = np.full((len(part), 3), 1/3.0)
    valid = np.all(np.isfinite(odds) & (odds > 1.0), axis=1)
    impl = 1.0 / odds[valid]
    impl = impl / impl.sum(axis=1, keepdims=True)
    probs[valid] = np.column_stack([impl[:, 2], impl[:, 1], impl[:, 0]])
    return probs

def brier(y, p):
    return float(np.mean([brier_score_loss((y == klass).astype(int), p[:, i]) for i, klass in enumerate(CLASS_LABELS)]))

def metrics(model, fold, y, p):
    pred = np.argmax(p, axis=1)
    per = f1_score(y, pred, labels=CLASS_LABELS, average=None, zero_division=0)
    return {'model': model, 'fold': fold, 'n_matches': len(y), 'accuracy': accuracy_score(y, pred),
            'f1_macro': f1_score(y, pred, average='macro', zero_division=0),
            'f1_weighted': f1_score(y, pred, average='weighted', zero_division=0),
            'log_loss': log_loss(y, p, labels=CLASS_LABELS), 'brier': brier(y, p),
            'away_f1': per[0], 'draw_f1': per[1], 'home_f1': per[2]}

def prep(train, test):
    cut = int(len(train) * 0.85)
    tr, val = train.iloc[:cut], train.iloc[cut:]
    imp = SimpleImputer(strategy='mean'); sc = StandardScaler()
    xtr = sc.fit_transform(imp.fit_transform(tr[feature_cols])).astype(np.float32)
    xv = sc.transform(imp.transform(val[feature_cols])).astype(np.float32)
    xte = sc.transform(imp.transform(test[feature_cols])).astype(np.float32)
    return xtr, tr['target_result'].to_numpy(int), xv, val['target_result'].to_numpy(int), xte, test['target_result'].to_numpy(int)

def train(model, xtr, ytr, xv, yv, lr=1e-3, wd=1e-4, dropout=0.3, max_epochs=500, patience=35):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=8, min_lr=1e-6)
    loss_fn = nn.CrossEntropyLoss()
    X = torch.tensor(xtr, device=DEVICE); y = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    Xv = torch.tensor(xv, device=DEVICE); yv_t = torch.tensor(yv, dtype=torch.long, device=DEVICE)
    best_state, best_loss, wait = None, 1e9, 0
    for epoch in range(1, max_epochs + 1):
        model.train(); perm = torch.randperm(len(X), device=DEVICE)
        for start in range(0, len(X), 64):
            idx = perm[start:start+64]
            opt.zero_grad(); loss = loss_fn(model(X[idx]), y[idx]); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad(): vloss = loss_fn(model(Xv), yv_t).item()
        sched.step(vloss)
        if vloss < best_loss - 1e-5:
            best_loss, wait = vloss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience and epoch >= 60: break
    if best_state is not None: model.load_state_dict(best_state)
    return model, best_loss

def predict(model, xte):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(torch.tensor(xte, device=DEVICE)), dim=1).cpu().numpy()

In [ ]:
seasons = sorted(df_feat['season_id'].unique())
hp_grid = [
    {'lr': 1e-3, 'wd': 1e-4, 'dropout': 0.30},
    {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.35},
    {'lr': 1e-3, 'wd': 1e-5, 'dropout': 0.25},
]
fold_rows, pred_rows = [], []
all_true, all_probs, all_market = [], {m: [] for m in MODELS}, []
start_time = time.time()

for fold_i, test_season in enumerate(seasons[1:], 1):
    train_df = df_feat[df_feat['season_id'] < test_season].copy()
    test_df = df_feat[df_feat['season_id'] == test_season].copy()
    xtr, ytr, xv, yv, xte, yte = prep(train_df, test_df)
    all_true.append(yte)
    all_market.append(market_probs(test_df))
    print(f'Fold {fold_i}: train < {test_season} ({len(train_df)}), test {test_season} ({len(test_df)})')
    for name, builder in MODELS.items():
        best = None
        for hp in hp_grid:
            torch.manual_seed(SEED)
            model = builder(xtr.shape[1], dropout=hp['dropout'])
            model, vloss = train(model, xtr, ytr, xv, yv, lr=hp['lr'], wd=hp['wd'], dropout=hp['dropout'])
            prob = predict(model, xte)
            row = metrics(name, test_season, yte, prob)
            row.update(hp); row['val_loss'] = vloss
            if best is None or vloss < best['val_loss']:
                best = {**row, 'prob': prob}
        prob = best.pop('prob')
        fold_rows.append(best); all_probs[name].append(prob)
        print(f"  {name}: acc={best['accuracy']:.3f}, macro={best['f1_macro']:.3f}, draw={best['draw_f1']:.3f}, ll={best['log_loss']:.3f}")

y_all = np.concatenate(all_true)
agg = [metrics(name, 'ALL', y_all, np.vstack(chunks)) for name, chunks in all_probs.items()]
mkt = metrics('bet365_closing', 'ALL', y_all, np.vstack(all_market)); mkt['source'] = 'market'
agg.append(mkt)
agg_df = pd.DataFrame(agg).sort_values('accuracy', ascending=False)
fold_df = pd.DataFrame(fold_rows)
runtime = {'torch_version': torch.__version__, 'cuda_available': torch.cuda.is_available(),
           'device': str(DEVICE), 'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           'runtime_seconds': round(time.time() - start_time, 2)}
fold_df.to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)
agg_df.to_csv(OUTPUT_DIR / 'model_comparison_walkforward.csv', index=False)
(OUTPUT_DIR / 'runtime.json').write_text(json.dumps(runtime, indent=2))
print('\nAggregate:')
display(agg_df[['model','accuracy','f1_macro','draw_f1','log_loss','brier']])
print(runtime)